# Progressive edge-growth algorithm 

## Definitions and notations:
$$ H \in \{0, 1\}^{m \times n} $$
$$ \text{Tanner graph is denoted as } G = (V, E) $$
$$ V = V_с \cup V_s \text{ --- set of nodes}$$
$$ V_c = \{c_0, c_1, \dots, c_{m - 1}\} \text{ --- set of check nodes} $$
$$ V_s = \{s_0, s_1, \dots, s_{n - 1}\} \text{ --- set of symbol (variable) nodes} $$
$$ E \subseteq V_c \times V_s  \text{ --- set of edges } $$
$$ (c_i, s_j) \in E \text{ if and only if } h_{i,j} \neq 0 $$
$$ D_s = \{d_{s_0}, d_{s_1}, \dots, d_{s_{n - 1}}\} \text{ --- symbol degree sequence} $$
$$ D_c = \{d_{c_0}, d_{c_1}, \dots, d_{c_{m - 1}}\} \text{ --- party-check degree sequence}$$
$$ \forall \text{ regular } G, \forall i, j \in \{0, 1, \dots, m - 1\} : d_{c_i} = d_{c_j} $$
$$ g_{s_j} = d(s_j, s_j) \text{ --- shortest path between } s_j \text{ and } s_j $$
$$ g = min_j \{g_{s_j}\} \text{ --- grith }$$
Tanner graphs characterized by degree distribution pairs.

The degree distribution for symbol nodes is defined as:
$$ \Lambda(x) = \sum_{i \geq 2}^{d_s^{max}} \Lambda_i x^i $$
where $\Lambda_i$ is fraction of symbol nodes connected to exactly $i$ check nodes. $d_s^{max} = max \{D_s\}$
$$ \sum_{i \geq 2}^{d_s^{max}} \Lambda_i = 1 $$
Similarly, the degree distribution of check nodes is define as
$$ \Phi(x) = \sum_{i \geq 2}^{d_c^{max}} \Phi_i x^i $$
where $\Phi_i$ is the fraction of parity-check nodes connected to exactly $i$ symbol nodes
$$ \sum_{i \geq 2}^{d_c^{max}} \Phi_i = 1 $$
$$ \mathcal{N}_{s_j}^l = \{s_i | d(s_i, s_j) \leq 2l + 1\}$$
$$ \bar{\mathcal{N}}_{s_j}^l = V_c \setminus \mathcal{N}_{s_j}^l $$
$$ \mathcal{N}_{с_j}^l = \{с_i | d(с_i, с_j) \leq 2l + 1\}$$
$$ \bar{\mathcal{N}}_{s_j}^l = V_c \setminus \mathcal{N}_{s_j}^l $$

In [9]:
import numpy as np

class PEG:
    def __init__(self, m, n, l):
        self.n = n
        self.m = m
        self.l = l
        self.H = np.zeros((m, n), dtype=np.int8)
        self.D_s = np.zeros(n, dtype=np.int16)
        self.D_c = np.zeros(m, dtype=np.int16)
        self.D_s_future = np.zeros(n, dtype=np.int16) + l

    def bfs(self, s):
        s_list = np.zeros(n, dtype = np.int8)
        s_list[s] = 1
        cur_c_list = np.zeros(m, dtype=np.int8)
        new_c_list = np.zeros(m, dtype=np.int8)
        c_Q = np.array((), dtype=np.int16)
        s_Q = np.array([s], dtype=np.int16)
        while(True):
            is_cycle = True
            if s_Q.size:
                for _s in s_Q:
                    cur_c_list_upd = np.argwhere(self.H[:, _s] == cur_c_list + 1).ravel()
                    new_c_list[self.H[:, _s] == 1] = 1
                    if cur_c_list_upd.size:
                        c_Q = np.hstack((c_Q, cur_c_list_upd)).astype(np.int16)
                        is_cycle = False
            s_Q = np.array(())
            if c_Q.size:
                for _c in c_Q:
                    s_list_update = np.argwhere(self.H[_c, :] == s_list + 1).ravel()
                    s_list[self.H[_c, :] == 1] = 1
                    if s_list_update.size:
                        s_Q = np.hstack((s_Q, s_list_update)).astype(np.int16)
            c_Q = np.array(())
            if is_cycle or np.count_nonzero(new_c_list) == m:
                ind = np.argwhere(cur_c_list == 0).ravel()
                degree = self.D_c[ind]
                return ind[np.argmin(degree)]
            cur_c_list = new_c_list.copy()

    def peg(self):
        for j in range(self.n):
            for k in range(self.D_s_future[j]):
                if k == 0:
                    c = np.argmin(self.D_c)
                else:
                    c = self.bfs(j)
                self.H[c, j] = 1
                self.D_s[j] += 1
                self.D_c[c] += 1
        np.savetxt(f'data/H_matrix/H{l}_{m}x{n}.txt', self.H, fmt='%d', delimiter=',')

n = 150
m = 100
l = 5
PEG(m, n, l).peg()
